# s07 - Multi-seed + confidence interval: ketahanan R1-R3 (SURVEY)

**Tujuan:** memberi dasar statistik untuk klaim R1-R3 — rerata, standar deviasi,
dan CI95 lintas 5 seed {13,42,101,202,303}, plus uji signifikansi federated vs
local-only (paired t + Wilcoxon). Pola warisi Paper 2/4.

**Metodologi:** simulasi FedAvg cross-dataset (identik s03/s06) diulang 5 seed.
Local-only sebagai pembanding. **Output:** `survey_out/r_multiseed_ci.csv`,
`r_significance.csv`.

**Prasyarat:** partisi s01 di S3.

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','scipy','scikit-learn','boto3'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, io, numpy as np
from sklearn.metrics import matthews_corrcoef
from scipy import stats
OUTDIR='survey_out'; os.makedirs(OUTDIR,exist_ok=True)
BUCKET='ssh-detection-features-232032302717'; REGION='ap-southeast-1'; BASE='unsw-far/federated_survey'
import boto3; s3=boto3.client('s3',region_name=REGION)
def load_npz_s3(key):
    try: obj=s3.get_object(Bucket=BUCKET,Key=key)
    except Exception as e: print('MISS',key,e); return None
    return np.load(io.BytesIO(obj['Body'].read()),allow_pickle=True)
SEEDS=[13,42,101,202,303]; ROUNDS=100; LOCAL_EPOCHS=5; LR=3e-3; MU=0.0

In [ ]:
P={}
for nm in ['cic_train','cic_test','unsw_train','unsw_test']:
    sub='cic' if nm.startswith('cic') else 'unsw'
    d=load_npz_s3(f'{BASE}/data/{sub}/{nm}.npz'); assert d is not None,'jalankan s01 dulu'
    P[nm]=(np.asarray(d['X'],float),np.asarray(d['y'],int))
Xg_te=np.vstack([P['cic_test'][0],P['unsw_test'][0]]); yg_te=np.concatenate([P['cic_test'][1],P['unsw_test'][1]])
Xtr_all=np.vstack([P['cic_train'][0],P['unsw_train'][0]]); LAYERS=[Xg_te.shape[1],128,64,1]
def init_w(seed):
    rng=np.random.RandomState(seed); W=[]
    for a,b in zip(LAYERS[:-1],LAYERS[1:]): W.append(rng.randn(a,b)*np.sqrt(2/a)); W.append(np.zeros(b))
    return W
relu=lambda x:np.maximum(0,x); sig=lambda x:1/(1+np.exp(-np.clip(x,-30,30)))
def fwd(W,X):
    a1=X@W[0]+W[1]; z1=relu(a1); a2=z1@W[2]+W[3]; z2=relu(a2); a3=z2@W[4]+W[5]
    return sig(a3).ravel(),(X,a1,z1,a2,z2)
def train_local(W,X,y,epochs,lr,mu,Wg,seed):
    rng=np.random.RandomState(seed); n=len(y); W=[w.copy() for w in W]
    for _ in range(epochs):
        idx=rng.permutation(n)
        for s in range(0,n,256):
            bi=idx[s:s+256]; Xb=X[bi]; yb=y[bi].astype(float)
            o,(X_,a1,z1,a2,z2)=fwd(W,Xb); g=(o-yb)/max(len(yb),1)
            gW4=z2.T@g[:,None]; gb4=g.sum(0); d2=(g[:,None]@W[4].T)*(a2>0)
            gW2=z1.T@d2; gb2=d2.sum(0); d1=(d2@W[2].T)*(a1>0); gW0=X_.T@d1; gb0=d1.sum(0)
            for i,gv in enumerate([gW0,gb0,gW2,gb2,gW4,gb4]):
                gv=np.asarray(gv).reshape(W[i].shape)
                if mu>0: gv=gv+mu*(W[i]-Wg[i])
                W[i]=W[i]-lr*gv
    return W
def fedavg(Ws,ns): tot=sum(ns); return [sum((ns[k]/tot)*Ws[k][i] for k in range(len(Ws))) for i in range(len(Ws[0]))]
def zfit(X): m=X.mean(0); s=X.std(0); s[s==0]=1; return m,s
m_mon,s_mon=zfit(Xtr_all); Xg_te_z=(Xg_te-m_mon)/s_mon
clients={'cic':((P['cic_train'][0]-m_mon)/s_mon,P['cic_train'][1]),
         'unsw':((P['unsw_train'][0]-m_mon)/s_mon,P['unsw_train'][1])}

## 1. 5 seed: federated vs local-only (MCC global)

In [ ]:
def fed(seed):
    W=init_w(seed)
    for t in range(ROUNDS):
        Ws=[];ns=[]
        for cid,(Xc,yc) in clients.items(): Ws.append(train_local(W,Xc,yc,LOCAL_EPOCHS,LR,MU,W,seed)); ns.append(len(yc))
        W=fedavg(Ws,ns)
    o,_=fwd(W,Xg_te_z); return matthews_corrcoef(yg_te,(o>=0.5).astype(int))
def local_best(seed):
    best=-1
    for cid,(Xc,yc) in clients.items():
        W=train_local(init_w(seed),Xc,yc,20,LR,0.0,None,seed)
        o,_=fwd(W,Xg_te_z); best=max(best,matthews_corrcoef(yg_te,(o>=0.5).astype(int)))
    return best
fed_vals=[fed(s) for s in SEEDS]; loc_vals=[local_best(s) for s in SEEDS]
print('federated',[round(v,3) for v in fed_vals]); print('local-only',[round(v,3) for v in loc_vals])

## 2. CI95 + uji signifikansi

In [ ]:
def ci95(v):
    v=np.array(v); m=v.mean(); se=v.std(ddof=1)/np.sqrt(len(v)); h=se*stats.t.ppf(0.975,len(v)-1)
    return m,v.std(ddof=1),(m-h,m+h)
fm,fsd,fci=ci95(fed_vals); lm,lsd,lci=ci95(loc_vals)
t_p=stats.ttest_rel(fed_vals,loc_vals).pvalue
try: w_p=stats.wilcoxon(fed_vals,loc_vals).pvalue
except Exception: w_p=float('nan')
print(f'federated  MCC {fm:.3f}+/-{fsd:.3f} CI95[{fci[0]:.3f},{fci[1]:.3f}]')
print(f'local-only MCC {lm:.3f}+/-{lsd:.3f} CI95[{lci[0]:.3f},{lci[1]:.3f}]')
print(f'paired-t p={t_p:.4f} | Wilcoxon p={w_p:.4f} | mean_diff={fm-lm:+.3f}')

In [ ]:
import csv
p1=os.path.join(OUTDIR,'r_multiseed_ci.csv')
with open(p1,'w',newline='') as f:
    w=csv.writer(f); w.writerow(['setting','mcc_mean','mcc_std','ci95_lo','ci95_hi','seeds'])
    w.writerow(['federated',fm,fsd,fci[0],fci[1],str(SEEDS)])
    w.writerow(['local-only',lm,lsd,lci[0],lci[1],str(SEEDS)])
p2=os.path.join(OUTDIR,'r_significance.csv')
with open(p2,'w',newline='') as f:
    w=csv.writer(f); w.writerow(['mean_diff','paired_t_p','wilcoxon_p']); w.writerow([fm-lm,t_p,w_p])
print('saved',p1,p2)
for p,key in [(p1,f'{BASE}/results/r_multiseed_ci.csv'),(p2,f'{BASE}/results/r_significance.csv')]:
    try: s3.upload_file(p,BUCKET,key)
    except Exception as e: print('(upload',p,'dilewati:',e,')')
print('=== s07 SELESAI. Pakai CI & p-value utk memperkuat klaim R1-R3 di §4 & §6. ===')